In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sqlalchemy import create_engine
from urllib.parse import quote_plus

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

In [4]:
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "uddy.ogkdfmkybqtrsglcizzt"
password = quote_plus("Uddodirim123")

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True,
    connect_args={"options": "-c statement_timeout=300000"}
)

test = pd.read_sql("SELECT current_user, current_database()", engine)
print("Connected as:", test.iloc[0, 0], "| DB:", test.iloc[0, 1])

Connected as: uddy | DB: postgres


In [7]:
# fact_sales — sampled at 10% for speed, enriched with calendar fields
df_sales = pd.read_sql("""
    SELECT
        s.date,
        s.store_id,
        s.product_id,
        s.units_sold,
        s.stockout_occurred,
        s.promo_flag,
        s.starting_inventory,
        s.ending_inventory,
        s.restriction_active,
        s.restriction_type,
        s.category,
        s.store_type,
        s.store_size,
        c.month,
        c.day_of_week,
        c.is_weekend,
        c.is_holiday,
        c.is_payday,
        c.season,
        c.is_black_friday_period
    FROM core.fact_sales s
    JOIN core.dim_calendar c ON s.date = c.date
""", engine, parse_dates=['date'])

# fact_promotions — full (small table)
df_promos = pd.read_sql("""
    SELECT date, store_id, product_id, promo_type, discount_pct, promo_flag
    FROM core.fact_promotions
    ORDER BY date
""", engine, parse_dates=['date'])

# fact_restriction_events — full
df_restrictions = pd.read_sql("""
    SELECT date, store_id, product_id, restriction_type, restriction_severity, duration_days
    FROM core.fact_restriction_events
    ORDER BY date
""", engine, parse_dates=['date'])

print(f"fact_sales sample:        {df_sales.shape[0]:,} rows")
print(f"fact_promotions:          {df_promos.shape[0]:,} rows")
print(f"fact_restriction_events:  {df_restrictions.shape[0]:,} rows")
print(f"\nDate range: {df_sales['date'].min().date()} → {df_sales['date'].max().date()}")

fact_sales sample:        1,642,200 rows
fact_promotions:          380 rows
fact_restriction_events:  22,136 rows

Date range: 2021-01-01 → 2026-05-09


In [8]:
df_sales.shape

(1642200, 20)

In [9]:
cutoff_date = "2026-04-30"

# fact_sales — full dataset with calendar enrichment
df_sales = pd.read_sql(f"""
    SELECT
        s.date,
        s.store_id,
        s.product_id,
        s.units_sold,
        s.stockout_occurred,
        s.promo_flag,
        s.starting_inventory,
        s.ending_inventory,
        s.restriction_active,
        s.restriction_type,
        s.category,
        s.store_type,
        s.store_size,
        c.month,
        c.day_of_week,
        c.is_weekend,
        c.is_holiday,
        c.is_payday,
        c.season,
        c.is_black_friday_period
    FROM core.fact_sales s
    JOIN core.dim_calendar c
        ON s.date = c.date
    WHERE s.date <= '{cutoff_date}'
""", engine, parse_dates=['date'])


# fact_promotions
df_promos = pd.read_sql(f"""
    SELECT
        date,
        store_id,
        product_id,
        promo_type,
        discount_pct,
        promo_flag
    FROM core.fact_promotions
    WHERE date <= '{cutoff_date}'
    ORDER BY date
""", engine, parse_dates=['date'])


# fact_restriction_events
df_restrictions = pd.read_sql(f"""
    SELECT
        date,
        store_id,
        product_id,
        restriction_type,
        restriction_severity,
        duration_days
    FROM core.fact_restriction_events
    WHERE date <= '{cutoff_date}'
    ORDER BY date
""", engine, parse_dates=['date'])


print(f"fact_sales:               {df_sales.shape[0]:,} rows")
print(f"fact_promotions:          {df_promos.shape[0]:,} rows")
print(f"fact_restriction_events:  {df_restrictions.shape[0]:,} rows")

print(f"\nDate range: {df_sales['date'].min().date()} → {df_sales['date'].max().date()}")

fact_sales:               1,634,640 rows
fact_promotions:          380 rows
fact_restriction_events:  22,041 rows

Date range: 2021-01-01 → 2026-04-30


In [10]:
cutoff_date = "2026-04-30"
df_sales = pd.read_sql(f"""
    SELECT *

    FROM core.fact_sales s
    JOIN core.dim_calendar c
        ON s.date = c.date
    WHERE s.date <= '{cutoff_date}'
""", engine, parse_dates=['date'])